# 06 - Visualisations (report figures)

Seven essential figures, one per analytical area, rendered inline and saved as PNG
in the repo's `figures/` directory. Everything is read from the curated artifacts -
no model is retrained and no Spark session is needed.

| figure | area |
|---|---|
`01_data_quality_and_demand.png` | column coverage + games by score range |
`02_models.png` | model error, R², feature importance |
`03_fps_curve.png` | FPS error per card + error on unseen cards |
`04_gpu_catalog.png` | benchmark spread by GPU type |
`05_clusters.png` | BIC choice, demand levels, level mix |
`06_rules.png` | strongest rules by lift |
`07_demand_trend.png` | minimum vs recommended score, game mix |
`08_fps_to_score.png` | target FPS -> required score for all games (1080p) |

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

DATA = Path(os.environ.get('DATA_DIR', '/home/jovyan/data'))
CUR = DATA / 'curated'
FIG = Path(os.environ.get('FIGURES_DIR', '/home/jovyan/figures'))
FIG.mkdir(parents=True, exist_ok=True)

PALETTE = ['#4f8cff', '#7c5cff', '#3ecf8e', '#f6c445', '#ff6b6b', '#9aa7c0']
# distinct colours for the demand levels (8+), no repeats
LEVEL_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#17becf']
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 9,
    'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
})


def save(fig, name):
    fig.tight_layout()
    path = FIG / name
    fig.savefig(path, bbox_inches='tight')
    print(f'saved {path} ({path.stat().st_size // 1024} KB)')

In [ ]:
registry = json.loads((CUR / 'model_selection.json').read_text())
clusters_meta = json.loads((CUR / 'analytics_clusters.json').read_text())
rules_meta = json.loads((CUR / 'analytics_rules.json').read_text())
clusters = pd.read_csv(CUR / 'game_clusters.csv')
trend = pd.read_csv(CUR / 'analytics_trend.csv')
# keep 'n/a' as a literal availability value (pandas would read it as NaN)
cards = pd.read_csv(CUR / 'gpus_classified.csv', keep_default_na=False, na_values=[''])
raw = pd.read_csv(DATA / 'games.csv')

print(f"games.csv {len(raw):,} rows x {len(raw.columns)} cols")
print(f"clusters {len(clusters):,} rows | trend {len(trend)} buckets | catalog {len(cards):,} cards")
print('registry sections:', sorted(registry))

## 1. Data quality and demand

In [ ]:
COLUMN_WORDS = {
    'name': 'Title', 'steam_appid': 'Steam ID', 'page_url': 'Page URL',
    'release_year': 'Release year', 'engine_fps_cap': 'Frame cap',
    'min_cpu': 'Minimum CPU', 'rec_cpu': 'Recommended CPU',
    'min_gpu': 'Minimum GPU', 'rec_gpu': 'Recommended GPU',
    'min_ram_gb': 'Minimum RAM', 'rec_ram_gb': 'Recommended RAM',
    'min_vram_gb': 'Minimum VRAM', 'rec_vram_gb': 'Recommended VRAM',
    'min_ghz': 'Minimum clock', 'rec_ghz': 'Recommended clock',
    'min_gpu_g3d': 'Minimum GPU score', 'rec_gpu_g3d': 'Recommended GPU score',
}


def column_word(name):
    if name in COLUMN_WORDS:
        return COLUMN_WORDS[name]
    if name.startswith('fps_1080p_high_'):
        return 'FPS: ' + name.replace('fps_1080p_high_', '').replace('_', ' ')
    return name.replace('_', ' ')


fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))

coverage = raw.replace(r'^\s*$', np.nan, regex=True).isna().mean().mul(100).sort_values()
axes[0].barh([column_word(c) for c in coverage.index], coverage.values, color=PALETTE[0])
axes[0].set_xlabel('Missing (%)')
axes[0].set_title(f'Column coverage (n={len(raw):,})')
axes[0].tick_params(axis='y', labelsize=7)

# 2,500-point bins up to 30k, then an overflow bin (almost nothing above 30k)
bins = list(range(2500, 30001, 2500))
labels = [f'{lo / 1000:g}-{hi / 1000:g}k' for lo, hi in zip(bins[:-1], bins[1:])]
counts = (pd.cut(clusters['required_g3d'], bins=bins + [float('inf')], right=False,
                 labels=labels + ['30k+'])
          .value_counts().sort_index())
axes[1].bar([str(c) for c in counts.index], counts.values, color=PALETTE[1])
axes[1].set_title(f'Games by performance need (n={len(clusters):,})')
axes[1].set_xlabel('Score range')
axes[1].set_ylabel('Games')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)

save(fig, '01_data_quality_and_demand.png')

## 2. Tier models

In [ ]:
metrics = registry['metrics']
order = ['rf', 'gbt', 'ridge', 'nn']
names = ['Random forest', 'Gradient boosting', 'Ridge', 'Neural net']
rmse = [metrics[m]['rmse'] for m in order]
mae = [metrics[m]['mae'] for m in order]
r2 = [metrics[m]['r2'] for m in order]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
x = np.arange(len(order))
w = 0.38
axes[0].bar(x - w / 2, rmse, w, label='RMSE', color=PALETTE[0])
axes[0].bar(x + w / 2, mae, w, label='MAE', color=PALETTE[1])
axes[0].axhline(registry['mean_baseline_rmse'], ls='--', color=PALETTE[4], label='Mean baseline')
axes[0].set_xticks(x, names, rotation=20)
axes[0].set_ylabel('Points')
axes[0].set_title('Error (lower is better)')
axes[0].legend(fontsize=7)

axes[1].bar(x, r2, color=PALETTE[2])
axes[1].set_xticks(x, names, rotation=20)
axes[1].set_ylim(0, 1)
axes[1].set_title('Fit (R²)')

importances = registry.get('feature_importances', {}).get('rf', {})
if importances:
    series = pd.Series(importances).sort_values()
    axes[2].barh([k.replace('_', ' ') for k in series.index], series.values, color=PALETTE[0])
    axes[2].set_title('Feature importance')
    axes[2].tick_params(axis='y', labelsize=8)
else:
    axes[2].text(0.5, 0.5, 'not recorded', ha='center', va='center')
    axes[2].set_axis_off()

save(fig, '02_models.png')

## 3. FPS curve

In [ ]:
curve = registry['fps_curve']
logo = sorted(curve['logo'].items(), key=lambda kv: kv[1]['mae'])
CARD_WORDS = {
    'iris_xe': 'Iris Xe', 'radeon_780m': 'Radeon 780M', 'vega_8': 'Vega 8',
    'gtx_1050': 'GTX 1050', 'gtx_1650': 'GTX 1650', 'rx_580': 'RX 580',
    'gtx_1080': 'GTX 1080', 'rtx_2060': 'RTX 2060', 'rtx_3060': 'RTX 3060',
    'rx_6700_xt': 'RX 6700 XT', 'rtx_3070': 'RTX 3070', 'rtx_4090': 'RTX 4090',
}
card_names = [CARD_WORDS.get(k, k.replace('_', ' ')) for k, _ in logo]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
axes[0].bar(card_names, [v['mae'] for _, v in logo], color=PALETTE[0])
axes[0].set_title('Error per card (left out)')
axes[0].set_ylabel('Error (fps)')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
for i, (_, v) in enumerate(logo):
    axes[0].text(i, v['mae'], f"{v['mae']:.0f}", ha='center', va='bottom', fontsize=6)

scores = [curve['curve_holdout_mae'], curve['rf_holdout']['mae']]
axes[1].bar(['Curve', 'Random forest'], scores, color=[PALETTE[0], PALETTE[2]])
axes[1].set_title('Error on unseen cards')
axes[1].set_ylabel('Error (fps)')
for i, v in enumerate(scores):
    axes[1].text(i, v, f'{v:.1f}', ha='center', va='bottom')

save(fig, '03_fps_curve.png')

## 4. GPU catalog

In [ ]:
KEEP = ['desktop', 'laptop', 'integrated']
present = [ff for ff in KEEP if ff in set(cards['form_factor'])]
groups = [cards.loc[cards['form_factor'] == ff, 'g3d_mark'].dropna().values for ff in present]

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.boxplot(groups, showfliers=False, patch_artist=True,
           boxprops={'facecolor': PALETTE[0] + '55', 'edgecolor': PALETTE[0]},
           medianprops={'color': PALETTE[1], 'linewidth': 1.6})
ax.set_xticks(range(1, len(present) + 1), [f'{ff.title()} (n={len(g):,})' for ff, g in zip(present, groups)])
ax.set_title('Benchmark score by GPU type')
ax.set_ylabel('Score')

accuracy = (registry.get('gpu_catalog') or {}).get('holdout_accuracy')
if accuracy is not None:
    ax.text(0.99, 1.02, f'Classifier accuracy {accuracy:.2f}', transform=ax.transAxes,
            ha='right', fontsize=8, color='#555')

save(fig, '04_gpu_catalog.png')

## 5. Demand clusters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
k_table = pd.DataFrame(clusters_meta['k_table'])
chosen = clusters_meta['chosen_k']['k']

axes[0].plot(k_table['k'], k_table['gm_bic'], 'o-', color=PALETTE[1], label='BIC')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v / 1e6:.1f}M'))
axes[0].set_xlabel('Levels')
axes[0].set_ylabel('BIC (lower is better)')
axes[0].set_title('Choosing the number of levels')
axes[0].axvline(chosen, color=PALETTE[4], ls=':', label=f'Chosen k={chosen}')
twin = axes[0].twinx()
twin.plot(k_table['k'], k_table['km_silhouette'], 's--', color=PALETTE[0], alpha=0.6, label='Silhouette')
twin.set_ylabel('Silhouette (higher is better)', rotation=270, labelpad=16, fontsize=8)
twin.tick_params(labelsize=7)
handles, labels = axes[0].get_legend_handles_labels()
handles2, labels2 = twin.get_legend_handles_labels()
axes[0].legend(handles + handles2, labels + labels2, fontsize=6, loc='upper right')

levels = [c['label'] for c in clusters_meta['clusters']]
for i, label in enumerate(levels):
    subset = clusters[clusters['label'] == label]
    axes[1].scatter(subset['required_g3d'], subset['fps_1080p_high_iris_xe'],
                    s=5, alpha=0.25, color=LEVEL_COLORS[i % len(LEVEL_COLORS)], label=f'{label} ({len(subset):,})')
axes[1].set_xlabel('Performance score')
axes[1].set_ylabel('Integrated FPS')
axes[1].set_title('Demand by level')
axes[1].legend(fontsize=6, markerscale=3)

released = clusters.dropna(subset=['release_year']).copy()
released['period'] = (released['release_year'] // 5 * 5).astype(int)
mix = (pd.crosstab(released['period'], released['label'], normalize='index')
       .mul(100).reindex(columns=levels).sort_index())
mix.plot(kind='bar', stacked=True, width=0.9, ax=axes[2], legend=False,
         color=[LEVEL_COLORS[i % len(LEVEL_COLORS)] for i in range(len(levels))])
axes[2].set_title('Level mix over time')
axes[2].set_xlabel('Period')
axes[2].set_ylabel('Share (%)')
axes[2].tick_params(axis='x', rotation=45, labelsize=7)
axes[2].legend(handles=axes[1].get_legend_handles_labels()[0],
               labels=levels, fontsize=5.5, ncol=4, loc='upper center',
               bbox_to_anchor=(0.5, -0.28), frameon=False)

save(fig, '05_clusters.png')

## 6. Association rules

In [ ]:
rows = rules_meta['top_rules'][:12]
fig, ax = plt.subplots(figsize=(11.5, 5.2))
ax.barh(range(len(rows)), [r['lift'] for r in rows], color=PALETTE[0])
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([', '.join(r['antecedent']) + ' → ' + ', '.join(r['consequent']) for r in rows], fontsize=7)
ax.invert_yaxis()
ax.set_xlabel('Lift')
ax.set_title(f"Top rules by lift ({rules_meta['rules_total']:,} rules)")
for i, r in enumerate(rows):
    ax.text(r['lift'], i, f" conf {r['confidence']:.2f}", va='center', fontsize=6)

top = rows[0]
fig.text(0.5, -0.04,
         'Lift = how much more likely the tier is given the specs than by chance (16 means 16x). '
         f'Top rule: {", ".join(top["antecedent"])} → {", ".join(top["consequent"])} '
         f'(confidence {top["confidence"]:.2f}).',
         ha='center', fontsize=7.5, color='#444')

save(fig, '06_rules.png')

## 7. Demand trend and catalog context

In [ ]:
periods = trend['period_start'].astype(str) + '-' + (trend['period_start'] + 4).astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].plot(periods, trend['median_min_g3d'], 'o-', color=PALETTE[0], label='Minimum')
axes[0].plot(periods, trend['median_rec_g3d'], 's-', color=PALETTE[1], label='Recommended')
axes[0].set_title('Score over time')
axes[0].set_xlabel('Period')
axes[0].set_ylabel('Score')
axes[0].tick_params(axis='x', rotation=45, labelsize=7)
axes[0].legend(fontsize=7, loc='upper left')

axes[1].plot(periods, trend['share_igpu_ok'] * 100, 'o-', color=PALETTE[2], label='Integrated ok')
axes[1].plot(periods, trend['share_capped'] * 100, 's-', color=PALETTE[3], label='Frame cap')
axes[1].plot(periods, trend['share_high_tier'] * 100, '^-', color=PALETTE[4], label='High tier')
axes[1].set_title('Game mix over time')
axes[1].set_xlabel('Period')
axes[1].set_ylabel('Share (%)')
axes[1].tick_params(axis='x', rotation=45, labelsize=7)
axes[1].legend(fontsize=7)

save(fig, '07_demand_trend.png')

## 8. FPS to required score

The recommender inverts each game's FPS curve; this is that relation across every
fitted game - the middle 80% band plus the median, the pooled prior, and three
notable titles.

Every measurement in the dataset is **1080p High**. Higher resolutions scale the
required score by a load factor in the API (1440p x1.54, 4K x2.83), not in this
model.

In [ ]:
curve_meta = json.loads((CUR / 'fps_curve.json').read_text())
curves = pd.read_csv(CUR / 'game_fps_model.csv')
curves = curves[(curves['source'] == 'game') & curves['a'].notna() & curves['b'].notna()]

fps_grid = np.linspace(30, 360, 60)
a = curves['a'].to_numpy()[:, None]
b = curves['b'].to_numpy()[:, None]
required = np.exp((np.log(fps_grid)[None, :] - a) / b)
p10, p50, p90 = np.percentile(required, [10, 50, 90], axis=0)

fig, ax = plt.subplots(figsize=(10.5, 5))
rng = np.random.default_rng(3208)
for i in rng.choice(len(curves), size=min(150, len(curves)), replace=False):
    ax.plot(fps_grid, required[i], color=PALETTE[5], alpha=0.06, lw=0.6)

ax.fill_between(fps_grid, p10, p90, color=PALETTE[0], alpha=0.16, label='Middle 80% of games')
ax.plot(fps_grid, p50, color=PALETTE[0], lw=2.2, label='Median game')

prior = np.exp((np.log(fps_grid) - float(curve_meta['prior_a'])) / float(curve_meta['prior_b']))
ax.plot(fps_grid, prior, '--', color=PALETTE[4], lw=1.4, alpha=0.9, label='Pooled prior')

HIGHLIGHT = [('cyberpunk 2077', 'Cyberpunk 2077'),
             ('red dead redemption 2', 'Red Dead Redemption 2'),
             ('counter-strike 2', 'Counter-Strike 2')]
lookup = {str(n).strip().lower(): (fa, fb) for n, fa, fb in zip(curves['name'], curves['a'], curves['b'])}
for i, (key, label) in enumerate(HIGHLIGHT):
    if key in lookup:
        fa, fb = lookup[key]
        ax.plot(fps_grid, np.exp((np.log(fps_grid) - fa) / fb),
                color=PALETTE[(i + 1) % len(PALETTE)], lw=1.5, label=label)

for fps in (60, 120):
    y = float(np.interp(fps, fps_grid, p50))
    ax.plot([fps], [y], 'o', color=PALETTE[0], ms=4)
    ax.annotate(f'{fps} FPS: {y:,.0f}', (fps, y), textcoords='offset points', xytext=(6, 8), fontsize=8)

catalog_max = int(cards['g3d_mark'].max())
CEILING = '#7f8a9d'
ax.axhline(catalog_max, color=CEILING, ls=':', lw=1.4,
           label=f'Best card ({catalog_max:,})')
ax.text(120, catalog_max * 0.955, 'above this line: no GPU in the catalogue',
        fontsize=7, va='top', color=CEILING)

ax.set_xlim(30, 360)
ax.set_ylim(0, catalog_max * 1.06)
ax.set_xticks([30, 60, 90, 120, 144, 240, 360])
ax.set_xlabel('Target FPS')
ax.set_ylabel('Required score')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v / 1000:.0f}k'))
ax.set_title(f'Target FPS to required score ({len(curves):,} games)')
ax.legend(fontsize=7, loc='upper left')

save(fig, '08_fps_to_score.png')

In [ ]:
figures = sorted(FIG.glob('*.png'))
for path in figures:
    print(f'{path.name:36} {path.stat().st_size / 1024:7.1f} KB')
print(f'{len(figures)} figures written to {FIG}')

## Publishing to the web UI

The figures and a small index (title + caption) are uploaded to the bucket so the
serving API can stream them (`GET /figures`, `GET /figures/{name}`) and the
frontend's **Insights** tab can display them.

In [ ]:
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import storage

# importance-first order; span=2 marks a featured card. Each featured figure is
# paired in the same grid row with the figure it depends on / complements.
GALLERY = [
    ('08_fps_to_score.png', 'FPS vs score',
     'Target FPS to required score across 17,265 games (1080p High); the band is the middle 80% and the dotted line is the best card available. 1440p/4K scale the requirement by 1.54x/2.83x.', 2,
     'How much benchmark score each frame-rate target needs. Every faint line is one game; the band holds the middle 80% of them. Read up from your FPS target to see the score a GPU must beat.'),
    ('03_fps_curve.png', 'FPS curve',
     'Error per left-out card and error on unseen cards; the curve beats the random forest 13.3 vs 49.0 fps.', 1,
     'How well the FPS model generalises: each bar hides one reference GPU from the fit and predicts it back. The right panel compares the curve with a random forest on cards neither has seen.'),
    ('05_clusters.png', 'Demand levels',
     'Number of levels chosen by BIC (3 seeds per k) and how the level mix shifts over time.', 2,
     'Games grouped by hardware demand. Panel 1 shows how the number of groups was chosen, panel 2 the groups themselves, panel 3 how the mix of groups changes with release era.'),
    ('07_demand_trend.png', 'Demand trend',
     'Median minimum and recommended GPU score per period, plus the shift in game mix.', 1,
     'How hardware demand grew: median minimum and recommended GPU score per 5-year era, and the share of games an integrated GPU can run, that hit the frame cap, or that need a high-tier card.'),
    ('02_models.png', 'Tier models',
     'Random forest is selected (R2 0.89, RMSE 1,372) against a mean baseline of 4,219.', 2,
     'Which model predicts the required score best. The random forest (selected) is compared with three alternatives on error, fit quality and the features it relies on.'),
    ('01_data_quality_and_demand.png', 'Data quality and demand',
     'Column coverage and games by score range (n=17,194).', 1,
     'What the source data actually contains: how much of each column is missing, and how the games spread across performance needs.'),
    ('04_gpu_catalog.png', 'GPU catalog',
     'Benchmark spread for desktop, laptop and integrated GPUs; classifier accuracy 0.67.', 1,
     'The GPU catalogue the recommender picks from: benchmark score spread for desktop, laptop and integrated cards.'),
    ('06_rules.png', 'Association rules',
     'Top rules by lift with confidence; lift 16 means the specs make the tier 16x more likely than chance.', 1,
     'Which hardware requirements travel together. Each bar is a rule such as "6-8 GB VRAM, no frame cap, recent release -> high tier", ranked by how much better than chance it predicts.'),
]

MODEL_BUCKET = os.environ.get('MODEL_BUCKET', 'gpu-recsys')
client = storage.Client()
try:
    bucket = client.get_bucket(MODEL_BUCKET)
except NotFound:
    try:
        bucket = client.create_bucket(MODEL_BUCKET)
    except Conflict:
        bucket = client.get_bucket(MODEL_BUCKET)

for name, title, caption, span, explanation in GALLERY:
    bucket.blob(f'models/figures/{name}').upload_from_filename(str(FIG / name), content_type='image/png')
bucket.blob('models/figures/index.json').upload_from_string(
    json.dumps([{'name': n, 'title': t, 'caption': c, 'span': sp, 'explanation': ex}
                for n, t, c, sp, ex in GALLERY], indent=2),
    content_type='application/json',
)
print(f'published {len(GALLERY)} figures + index.json to gs://{MODEL_BUCKET}/models/figures')